# Module 2.5 — Token Classification for Field Extraction (NER)
**DeskMate SLM Curriculum · Phase 2 · Notebook 12**

Run on **Google Colab** (CPU is fine; GPU ~3× faster).

By the end you will have:
- A fine-tuned MiniLM NER model extracting product, version, os from raw tickets
- Per-entity seqeval F1 (span-level, stricter than token F1)
- An inference function returning `{"product": ..., "version": ..., "os": ...}`
- Model saved to `models/field_extractor/`

Read `2.5_ner_extraction.md` for full theory.

---

## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q transformers==4.44.0 datasets==2.21.0 torch==2.3.1 seqeval==1.2.2

In [ ]:
import random, json, pathlib
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments, Trainer,
)
from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Runtime : {RUNTIME}')

## Step 2 — Label Schema

In [ ]:
LABELS   = ['O', 'B-product', 'I-product', 'B-version', 'I-version', 'B-os', 'I-os']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
N_LABELS = len(LABELS)

# Round-trip check
for l in LABELS:
    assert ID2LABEL[LABEL2ID[l]] == l

print(f'Label set ({N_LABELS} classes):')
for i, l in ID2LABEL.items():
    print(f'  {i}: {l}')

## Step 3 — Load Tokenizer

In [ ]:
MODEL_NAME = 'microsoft/MiniLM-L12-H384-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer: {MODEL_NAME}')
print(f'Vocab size: {tokenizer.vocab_size:,}')

## Step 4 — Load Synthetic Data

We need examples with `fields` containing character spans.
These come from `train_augmented.jsonl` produced in Module 2.2.
A placeholder is created if the file is missing.

In [ ]:
def make_placeholder_data(n=60):
    templates = [
        ('I cannot use {product} {version} on {os}.', 'DeskMate Pro', 'v2.3', 'Windows 10'),
        ('Bug in {product} {version}: crash on {os}.', 'AgentCore', 'v3.1', 'macOS'),
        ('{product} {version} is slow on {os}.', 'FlowPilot', 'v1.0', 'Ubuntu 22'),
    ]
    rows = []
    for i in range(n):
        tmpl, prod, ver, os_val = templates[i % len(templates)]
        text = tmpl.format(product=prod, version=ver, os=os_val)
        fields = {}
        for field_name, val in [('product', prod), ('version', ver), ('os', os_val)]:
            s = text.find(val)
            if s >= 0:
                fields[field_name] = {'start': s, 'end': s + len(val), 'value': val}
        rows.append({'text': text, 'intent': 'technical_bug', 'priority': 'high',
                     'source': 'placeholder', 'fields': fields})
    return rows

def load_split(name, require_fields=True):
    p = DATA_PROC / f'{name}.jsonl'
    if not p.exists():
        print(f'  WARNING: {p} missing — using placeholder (run Module 2.2 first)')
        rows = make_placeholder_data(60 if 'train' in name else 20)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text('\n'.join(json.dumps(r) for r in rows))
    df = pd.read_json(p, lines=True)
    if require_fields and 'fields' not in df.columns:
        df['fields'] = [{}] * len(df)
    # Keep only examples with at least one field
    if require_fields:
        df = df[df['fields'].apply(lambda f: isinstance(f, dict) and len(f) > 0)]
    return Dataset.from_pandas(df.reset_index(drop=True))

aug = DATA_PROC / 'train_augmented.jsonl'
raw = DatasetDict({
    'train': load_split('train_augmented' if aug.exists() else 'train'),
    'val':   load_split('val'),
    'test':  load_split('test'),
})
for split, ds in raw.items():
    print(f'  {split:<6}: {len(ds):>5} examples with fields')

In [ ]:
# Inspect one example
ex = raw['train'][0]
print('Text  :', ex['text'])
print('Fields:')
for fname, fval in ex.get('fields', {}).items():
    s, e, v = fval['start'], fval['end'], fval['value']
    print(f'  {fname:<10}: [{s}:{e}] {v!r}')
    assert ex['text'][s:e] == v, f'Span mismatch for {fname}'

## Step 5 — Character Spans → BIO Labels

We tokenize with `return_offsets_mapping=True` to get `(char_start, char_end)` per subword,
then use those offsets to assign BIO labels and align with `word_ids()`.

In [ ]:
def spans_to_bio(example):
    text   = example['text']
    fields = example.get('fields', {})

    # Build char-level field map: char_pos -> (field_name, is_start)
    char_field = {}   # start_char -> (field_name, end_char)
    for fname, fval in fields.items():
        s, e = fval['start'], fval['end']
        char_field[(s, e)] = fname

    enc = tokenizer(
        text,
        truncation=True,
        max_length=128,
        return_offsets_mapping=True,
    )
    offsets  = enc.pop('offset_mapping')  # remove — Trainer doesn't expect it
    word_ids = enc.word_ids()

    # Assign label per word (not subword); -100 for continuation and special tokens
    labels     = []
    prev_word  = None
    word_label = {}   # word_index -> BIO label string

    # First pass: determine label per word using char offsets
    for tok_idx, wid in enumerate(word_ids):
        if wid is None or wid in word_label:
            continue
        tok_start, tok_end = offsets[tok_idx]
        assigned = 'O'
        for (fs, fe), fname in char_field.items():
            if tok_start >= fs and tok_end <= fe:
                assigned = ('B-' if tok_start == fs else 'I-') + fname
                break
        word_label[wid] = LABEL2ID[assigned]

    # Second pass: expand to subword tokens
    for wid in word_ids:
        if wid is None:
            labels.append(-100)
        elif wid != prev_word:
            labels.append(word_label.get(wid, LABEL2ID['O']))
        else:
            labels.append(-100)
        prev_word = wid

    enc['labels'] = labels
    return enc

# Smoke test on first example
ex = raw['train'][0]
out = spans_to_bio(ex)
tokens = tokenizer.convert_ids_to_tokens(out['input_ids'])
print(f'Text: {ex["text"]}')
print(f'\n{"Token":<18} {"Label":<15} {"Attn"}')
print('-' * 45)
for tok, lbl, attn in zip(tokens, out['labels'], out['attention_mask']):
    lbl_str = ID2LABEL[lbl] if lbl != -100 else 'IGNORE'
    print(f'{tok:<18} {lbl_str:<15} {attn}')

## Step 6 — Tokenize & Align the Dataset

`batched=False` because `word_ids()` is per-sequence.

In [ ]:
# batched=False required for word_ids() alignment
drop_cols = [c for c in raw['train'].column_names
             if c not in ('input_ids','attention_mask','labels')]

tokenized = raw.map(
    spans_to_bio,
    batched=False,
    remove_columns=drop_cols,
    desc='Tokenizing + aligning',
)

print('Tokenized dataset:')
for split, ds in tokenized.items():
    print(f'  {split:<6}: {len(ds):>5} examples  columns: {ds.column_names}')

In [ ]:
# Label distribution (non-O, non-ignore)
from collections import Counter
cnts = Counter()
for ex in tokenized['train']:
    for lbl in ex['labels']:
        if lbl not in (-100, LABEL2ID['O']):
            cnts[ID2LABEL[lbl]] += 1
print('Field token counts in train:')
for lbl, cnt in sorted(cnts.items()):
    print(f'  {lbl:<15}: {cnt}')

## Step 7 — DataCollatorForTokenClassification

In [ ]:
collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)
print('Collator ready.')

# Show batch padding
tokenized.set_format('torch', columns=['input_ids','attention_mask','labels'])
sample = [tokenized['train'][i] for i in range(4)]
lens   = [len(s['input_ids']) for s in sample]
print(f'Sequence lengths before collation: {lens}')
batch  = collator(sample)
print(f'input_ids shape  : {tuple(batch["input_ids"].shape)}')
print(f'labels shape     : {tuple(batch["labels"].shape)}')
print(f'Labels row 0 (pad=-100): {batch["labels"][0].tolist()[:20]}...')

## Step 8 — Metrics (seqeval span-level F1)

seqeval reconstructs full spans before scoring — a span is correct only if both boundaries match.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    true_labels = [
        [ID2LABEL[l] for l in label_row if l != -100]
        for label_row in labels
    ]
    true_preds = [
        [ID2LABEL[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(preds, labels)
    ]
    f1 = seq_f1(true_labels, true_preds, average='macro', zero_division=0)
    return {'f1': float(f1)}

## Step 9 — Train the NER Model

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME}  ({n_params/1e6:.1f}M params)')

In [ ]:
ner_args = TrainingArguments(
    output_dir                  = str(MODELS_DIR / 'field_extractor'),
    num_train_epochs            = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = (device == 'cuda'),
    seed                        = SEED,
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = ner_args,
    train_dataset   = tokenized['train'],
    eval_dataset    = tokenized['val'],
    data_collator   = collator,
    compute_metrics = compute_metrics,
    tokenizer       = tokenizer,
)

print('Starting NER training ...')
trainer.train()
print('Done.')

In [ ]:
import matplotlib.pyplot as plt

logs      = [x for x in trainer.state.log_history if 'loss' in x and 'eval_loss' not in x]
eval_logs = [x for x in trainer.state.log_history if 'eval_f1' in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
if logs:
    ax1.plot([x['step'] for x in logs], [x['loss'] for x in logs], color='#4C72B0')
    ax1.set_xlabel('Step'); ax1.set_ylabel('Train loss')
    ax1.set_title('NER — Training Loss')
if eval_logs:
    ax2.plot([x['epoch'] for x in eval_logs], [x['eval_f1'] for x in eval_logs],
             'o-', color='#DD8452')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val seqeval F1')
    ax2.set_title('NER — Val F1')
    ax2.set_ylim(0, 1)
plt.tight_layout(); plt.savefig(str(MODELS_DIR / 'ner_training.png'), bbox_inches='tight')
plt.show()

## Step 10 — Evaluate (seqeval)

In [ ]:
test_preds = trainer.predict(tokenized['test'])
logits  = test_preds.predictions
labels  = test_preds.label_ids
p_ids   = logits.argmax(axis=-1)

true_labels = [
    [ID2LABEL[l] for l in row if l != -100]
    for row in labels
]
true_preds = [
    [ID2LABEL[p] for p, l in zip(p_row, l_row) if l != -100]
    for p_row, l_row in zip(p_ids, labels)
]

print('NER — test set (seqeval span-level):')
print(seq_report(true_labels, true_preds, zero_division=0))

## Step 11 — Inference Demo

In [ ]:
def extract_fields(text, model, tokenizer):
    model.eval()
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits   # (1, T, N_LABELS)
    pred_ids = logits[0].argmax(-1).tolist()
    tokens   = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    bio_seq  = [ID2LABEL[i] for i in pred_ids]

    result    = {}
    cur_field = None
    cur_toks  = []
    for token, label in zip(tokens, bio_seq):
        if token in ('[CLS]', '[SEP]', '[PAD]'):
            continue
        if label.startswith('B-'):
            if cur_field:
                result[cur_field] = tokenizer.convert_tokens_to_string(cur_toks).strip()
            cur_field = label[2:]; cur_toks = [token]
        elif label.startswith('I-') and cur_field and cur_field == label[2:]:
            cur_toks.append(token)
        else:
            if cur_field:
                result[cur_field] = tokenizer.convert_tokens_to_string(cur_toks).strip()
            cur_field = None; cur_toks = []
    if cur_field:
        result[cur_field] = tokenizer.convert_tokens_to_string(cur_toks).strip()
    return result

sample_tickets = [
    'I cannot use DeskMate Pro v2.3 on Windows 10, it keeps crashing.',
    'AgentCore v3.1 freezes on macOS Ventura every time I open a ticket.',
    'FlowPilot v1.0 is extremely slow on Ubuntu 22.',
    'The export button throws a 500 error.',
    'I forgot my password and cannot log in.',
]

print(f'{"Ticket":<55} {"Extracted fields"}')
print('-' * 90)
for ticket in sample_tickets:
    fields = extract_fields(ticket, trainer.model, tokenizer)
    print(f'{ticket[:54]:<55} {json.dumps(fields)}')

## Step 12 — Save Model

In [ ]:
save_path = MODELS_DIR / 'field_extractor'
trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))
print(f'NER model saved: {save_path}')

## Checkpoint

> **Why does subword tokenization complicate token-level labels, and how do you align them?**

Write your answer below. Cover:
1. What WordPiece does to multi-syllable words.
2. What label a continuation subword (`##mate`) should receive and why.
3. How `word_ids()` is used to implement the alignment.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| NER model | `models/field_extractor/` |
| Training chart | `models/ner_training.png` |

**What you've built:** three DeskMate components now exist — intent classifier,
priority classifier, and field extractor.

**Next:** Module 2.6 — Domain-specific evaluation. Measure all three models
on the frozen gold set with per-class F1, calibration curves, and error analysis.